# Xây dựng giao diện người dùng cơ bản cho agent

Trong bài học này, bạn sẽ kết nối một agent LangChain với giao diện chat React bằng cách sử dụng **CopilotKit** và giao thức **AG-UI** - đây cũng chính là mô hình tiêu chuẩn được sử dụng để kết nối bất kỳ backend agent nào với bất kỳ frontend nào.

## 📋 Mục tiêu học tập

1. **Chạy một agent LangChain** - Khởi động một backend tương thích với AG-UI bằng FastAPI.
2. **Thiết lập CopilotKit** - Kết nối frontend React với agent của bạn thông qua `CopilotRuntime`.
3. **Chuyển đổi các backend agent** - Chuyển đổi linh hoạt giữa LangChain/OpenAI và Google ADK/Gemini mà không cần thay đổi mã nguồn UI.

---

## Trước khi bắt đầu

Bài học này sử dụng lệnh `%%writefile` (trong môi trường Jupyter) để lưu mã nguồn trực tiếp vào thư mục dự án frontend trên ổ đĩa. Ứng dụng đang chạy sẽ tự động nhận diện những thay đổi đó - vì vậy bạn sẽ thấy giao diện người dùng của mình cập nhật theo thời gian thực khi bạn thực hành qua các bước.

> 💡 **Bạn đã hoàn thành bài học này trước đây?** 
> Nếu bạn muốn bắt đầu lại từ đầu, hãy chạy đoạn mã bên dưới để khôi phục tất cả các tệp về trạng thái gốc. **Hãy bỏ qua bước này nếu đây là lần đầu tiên bạn học.**

In [ ]:
# from helper import reset_lesson
# reset_lesson(2)

---

## Bạn sẽ xây dựng những gì?

Một giao diện chat được hỗ trợ bởi agent LangChain, kết nối thông qua CopilotKit. Đến cuối bài học, bạn sẽ có thể trò chuyện trực tiếp với agent của mình trên trình duyệt web:

> *Người dùng: Please write me a poem!*

![CopilotKit Chat](images/copilotkit-chat.png)

---

## Cài đặt các thư viện

Trong môi trường học tập này, tất cả các thư viện backend và frontend đã được cài đặt sẵn. Hãy chạy đoạn mã tiếp theo để đảm bảo mọi thứ đã sẵn sàng.

> ⚠️ **Nếu bạn đang chạy trên máy tính cá nhân?** 
> Hãy cài đặt các thư viện backend bằng lệnh `pip install -r requirements.txt`, sau đó chạy `npm install` trong thư mục `frontend/`.

In [1]:
# Tắt các thông báo cảnh báo để output gọn gàng hơn
import warnings
warnings.filterwarnings("ignore")

# !pip install -r requirements.txt

# Cài đặt các dependency cho frontend (sử dụng hàm helper)
from helper import install_frontend
install_frontend()

Installing frontend dependencies ...

up to date, audited 971 packages in 5s

249 packages are looking for funding
  run `npm fund` for details

34 vulnerabilities (7 low, 20 moderate, 7 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
npm warn allow-scripts 3 packages have install scripts not yet covered by allowScripts:
npm warn allow-scripts   @scarf/scarf@1.4.0 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.27.7 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.25.12 (install: (install scripts present))
npm warn allow-scripts
npm warn allow-scripts Run `npm approve-scripts --allow-scripts-pending` to review, or `npm approve-scripts <pkg>` to allow.
✓ Frontend dependencies installed


Bây giờ, bạn sẽ tải các API key cho các mô hình AI mà agent của bạn sẽ sử dụng.

In [2]:
from helper import load_api_keys
load_api_keys()

✓ OpenAI API key loaded
✓ Google API key loaded


> 🔑 **Lưu ý khi tự chạy trên máy tính của bạn:**
> Bạn sẽ cần chuẩn bị API key của riêng mình:
> - **OpenAI** (dùng trong phần bài học chính): [platform.openai.com/api-keys](https://platform.openai.com/api-keys)
> - **Google AI** (dùng trong phần bổ sung): [aistudio.google.com/apikey](https://aistudio.google.com/app/apikey)

---

## Xây dựng agent

### Khởi động server

CopilotKit kết nối với agent của bạn thông qua một HTTP endpoint tương thích với chuẩn **AG-UI**. Tại đây, bạn sẽ khởi động một server FastAPI và gắn `LangGraphAGUIAgent` vào đó.

In [3]:
from fastapi import FastAPI

# Các thư viện của CopilotKit và AG-UI dành cho server AG-UI
from ag_ui_langgraph import add_langgraph_fastapi_endpoint
from copilotkit import LangGraphAGUIAgent
from langchain.agents import create_agent

# Hàm helper đơn giản giúp khởi động server và quản lý xung đột cổng (port)
from helper import start_server

# Tích hợp endpoint AG-UI vào ứng dụng FastAPI
app = FastAPI()
graph = create_agent("openai:gpt-4.1") # Khởi tạo agent cơ bản
agent = LangGraphAGUIAgent(
    name="demo_agent",
    description="Demo agent",
    graph=graph,
)
add_langgraph_fastapi_endpoint(app=app, agent=agent, path="/")

# Khởi động server ở cổng 8002
start_server(app, port=8002)

✓ Server running at http://localhost:8002


> 💡 **AG-UI là gì?** 
> [AG-UI](https://docs.ag-ui.com) là một giao thức mở dùng để kết nối các backend agent với các frontend. CopilotKit sử dụng nó ở tầng dưới. Bạn sẽ tìm hiểu thêm về cách thức hoạt động của nó ở phần cuối của bài học này.

### Định nghĩa agent

Tiếp theo, bạn sẽ tạo một agent LangChain với mô hình OpenAI, bộ kiểm tra trạng thái bộ nhớ (memory checkpointer) và middleware của CopilotKit.

Bạn có thể chạy lại đoạn mã này bất cứ khi nào bạn muốn thay đổi cấu hình của agent - việc gán `agent.graph = ...` sẽ thực hiện tải lại (hot-reload) đồ thị mà không cần phải khởi động lại server.

In [4]:
from copilotkit import CopilotKitMiddleware

# Các thư viện của LangChain agent
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

# Tạo một LangChain agent
graph = create_agent(
    model=ChatOpenAI(model="gpt-4.1"),
    tools=[],
    middleware=[CopilotKitMiddleware()],
    checkpointer=MemorySaver(),
    system_prompt=("Bạn là một trợ lý hữu ích."),
)

# Cập nhật đồ thị của agent (hot reloads)
agent.graph = graph
print("✓ Đồ thị agent đã được cập nhật!")

✓ Đồ thị agent đã được cập nhật!


**Lưu ý quan trọng:** `CopilotKitMiddleware()` chính là cầu nối liên kết agent của bạn với CopilotKit - nó cho phép mô hình AI khám phá và gọi các công cụ ở frontend, điều mà bạn sẽ được thực hành xây dựng trong bài tiếp theo. Nếu không có middleware này, agent sẽ chỉ nhìn thấy các công cụ được định nghĩa ở phía backend.

---

## Bắt đầu với CopilotKit

CopilotKit cung cấp cả UI hoàn toàn linh hoạt (headless) và các React component được xây dựng sẵn cho giao diện của agent. Trong bài học này, bạn sẽ sử dụng ba thành phần chính:

- **`CopilotRuntime`**: Một cầu nối bảo mật kết nối frontend của bạn với bất kỳ backend agent nào.
- **`CopilotKit`**: React provider dùng để cấu hình kết nối runtime cho ứng dụng của bạn.
- **`CopilotChat`**: Một giao diện chat tùy chỉnh, tích hợp sẵn đầy đủ các tính năng cho agent của bạn.

### Khởi động frontend

Bạn sẽ sử dụng một ứng dụng đã được xây dựng sẵn làm bối cảnh cho các bài học. Khi bạn thực hiện thay đổi, nó sẽ tự động cập nhật và hiển thị ngay lập tức.

Hãy chạy hai đoạn mã tiếp theo để khởi động dev server và mở chế độ xem trước trực tiếp.

In [5]:
from helper import start_frontend
start_frontend(port=3002)

Starting frontend on port 3002 ...
✓ App running at http://localhost:3002

Read the logs: /home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/1-building-a-basic-agent-ui/frontend/dev-logs.txt


Cuối cùng, hiển thị ứng dụng:

In [ ]:
from helper import display_app
display_app(port=3002)

Tiếp theo, bạn sẽ thiết lập phần chat để trò chuyện với agent. Ban đầu, đây sẽ chỉ là một ứng dụng chat trống.

### Thiết lập `CopilotRuntime`

`CopilotRuntime` là cầu nối bảo mật giữa frontend và agent backend. Tại đây, bạn sẽ đăng ký agent LangChain của mình dưới dạng agent `default` (mặc định):

In [7]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import { LangGraphHttpAgent } from "@ag-ui/langgraph";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";

// Khai báo agent LangGraph đang chạy ở cổng 8002
const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

// Đăng ký agent vào CopilotRuntime
const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
  },
});

// Tạo endpoint cho CopilotKit
const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

// Chạy API server ở cổng 4002
serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


> 💡 **Tại sao lại dùng `/v2`?** 
> Các package của CopilotKit xuất ra cả API v1 và v2. Đường dẫn import `/v2` cung cấp cho bạn các hook và component mới nhất. Toàn bộ mã nguồn trong khóa học này đều sử dụng v2.

### Bao bọc ứng dụng trong provider `CopilotKit`

Provider `CopilotKit` kết nối ứng dụng React của bạn với runtime thông qua `runtimeUrl`. Hãy đặt nó trong tệp `main.tsx` để bao bọc toàn bộ ứng dụng - hoặc bạn cũng có thể bao bọc xung quanh từng component riêng lẻ nếu muốn.

In [8]:
%%writefile frontend/src/main.tsx

import { StrictMode } from "react";
import { createRoot } from "react-dom/client";
import { CopilotKit } from "@copilotkit/react-core/v2";
import "@copilotkit/react-core/v2/styles.css";
import "./globals.css";
import App from "./App";

createRoot(document.getElementById("root")!).render(
  <StrictMode>
    <main className="h-screen w-screen">
      <CopilotKit runtimeUrl="/api/copilotkit" useSingleEndpoint={false}>
        <App />
      </CopilotKit>
    </main>
  </StrictMode>,
);

Overwriting frontend/src/main.tsx


### Thiết lập component `CopilotChat`

Cuối cùng, thêm component `CopilotChat` và trỏ nó vào agent `default` mà bạn đã đăng ký trong `CopilotRuntime`.

In [9]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

const agentId = "default";

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


> 💡 **Lưu ý:** Bạn cũng có thể sử dụng CopilotKit ở chế độ "headless" (không có sẵn UI chat) thông qua hook `useAgent`. Trong bài học này, chúng ta sử dụng component `CopilotChat` có sẵn cho đơn giản.

### Trải nghiệm ngay!

Ứng dụng của bạn hiện đã có giao diện chat hoạt động. Hãy gọi hàm hiển thị một lần nữa để mở giao diện và thử trò chuyện với agent của bạn.

> *Bạn có thể thử nhắn: "Hello chat!"*

In [ ]:
from helper import display_app
display_app(port=3002)

---

## 🎁 Phần thưởng - Kết nối với Google ADK (Gemini)

Trong phần này, bạn sẽ thêm một agent backend thứ hai sử dụng hệ sinh thái **Google ADK** và mô hình **Gemini** - sau đó chuyển đổi sang backend này mà không cần thay đổi bất kỳ dòng mã UI nào.

> ⚠️ **Lưu ý:** Phần này yêu cầu `GEMINI_API_KEY`. Bạn có thể lấy Gemini API key của riêng mình từ [Google AI Studio](https://aistudio.google.com/app/apikey).

### Thiết lập agent ADK

Đầu tiên, khởi tạo và chạy Google ADK agent ở cổng `8009`:

In [10]:
from fastapi import FastAPI
from ag_ui_adk import ADKAgent, add_adk_fastapi_endpoint
from google.adk.agents import LlmAgent
from helper import start_server

# Khởi tạo mô hình Gemini
gemini_agent = LlmAgent(
    name="assistant",
    model="gemini-3.5-flash-lite",
    instruction="Bạn là một trợ lý hữu ích và vui vẻ!",
)

# Bao bọc vào ADKAgent
adk_agent = ADKAgent(
    adk_agent=gemini_agent,
    app_name="demo_app",
    user_id="demo_user",
    session_timeout_seconds=3600,
    use_in_memory_services=True,
)

app_adk = FastAPI()
add_adk_fastapi_endpoint(app_adk, adk_agent, path="/")

# Khởi động server cho Gemini
start_server(app_adk, port=8009)

✓ Server running at http://localhost:8009


### Cập nhật `CopilotRuntime` để đăng ký agent mới

Bây giờ, hãy cập nhật runtime để đăng ký thêm agent Gemini song song với agent mặc định (LangChain):

In [11]:
%%writefile frontend/server.ts

import { CopilotRuntime, createCopilotEndpoint } from "@copilotkit/runtime/v2";
import { HttpAgent } from "@ag-ui/client";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";
import { serve } from "@hono/node-server";

const langGraphAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8002",
});

const adkAgent = new HttpAgent({
  url: process.env.ADK_AGENT_URL || "http://localhost:8009",
});

const runtime = new CopilotRuntime({
  agents: {
    default: langGraphAgent,
    gemini: adkAgent, // Đăng ký thêm agent Gemini
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4002 }, () => {
  console.log("CopilotKit API server running at http://localhost:4002");
});

Overwriting frontend/server.ts


### Cập nhật frontend để sử dụng Gemini

Để thay đổi mô hình đang trò chuyện, bạn chỉ cần cập nhật tham số `agentId` trong tệp `frontend/src/App.tsx`:
- `"default"` -> Sử dụng LangChain/OpenAI
- `"gemini"` -> Sử dụng ADK/Gemini

In [12]:
%%writefile frontend/src/App.tsx

import { CopilotChat } from "@copilotkit/react-core/v2";

export const agentId = "gemini"; // Chuyển sang dùng Gemini

export default function App() {
  return <CopilotChat agentId={agentId} />;
}

Overwriting frontend/src/App.tsx


### Hiển thị ứng dụng

Ứng dụng chat của bạn lúc này đã được kết nối với backend Gemini. Hãy thử hỏi nó một vài câu hỏi và chú ý sự khác biệt trong cách phản hồi!

> *Bạn có thể hỏi: "Bạn đang chạy bằng mô hình nào?"*

In [ ]:
from helper import display_app
display_app(port=3002)

## AG-UI là gì?

**AG-UI** (Agent-User Interaction) là một giao thức mở, dựa trên sự kiện nhằm tiêu chuẩn hóa việc kết nối các agent backend với frontend. Nó chuẩn hóa luồng di chuyển của các tin nhắn chat, lệnh gọi công cụ, cập nhật trạng thái và stream token qua HTTP.

Bạn vừa thấy giao thức này hoạt động trong thực tế - đây là lý do tại sao nó lại quan trọng:
- CopilotKit có thể giao tiếp với *bất kỳ* backend nào triển khai chuẩn AG-UI.
- Bạn đã chuyển đổi từ LangChain/OpenAI sang ADK/Gemini chỉ bằng một thay đổi cấu hình nhỏ - không cần viết lại mã UI.
- Hành vi streaming (đổ chữ từ từ) và gọi tool được giữ nhất quán giữa các framework khác nhau.

![AG-UI protocol diagram](images/protocols.png)

---

## 🎯 Tổng kết những gì bạn đã học

- Cách chạy một agent LangChain phía sau một endpoint tương thích AG-UI và kết nối nó với CopilotKit.
- Cách tải lại nóng (hot-reload) đồ thị của agent trong quá trình phát triển.
- Cách một frontend duy nhất có thể chuyển đổi dễ dàng giữa các backend LangChain/OpenAI và ADK/Gemini.

## 🚀 Bước tiếp theo

Trong **bài tiếp theo**, bạn sẽ tập trung vào **giao diện người dùng tạo sinh kiểm soát**:
- Đăng ký các công cụ frontend định kiểu bằng hook `useComponent()`.
- Hiển thị kết quả đầu ra có cấu trúc của công cụ trực tiếp trong khung chat.
- Giữ cho hành vi của giao diện người dùng (được điều khiển bởi mô hình AI) có thể dự đoán được và an toàn.